# S5.1 — Unified LLM Client (Ollama + Gemini 3 Flash)

Interactive verification of the LLM provider abstraction layer.
Tests: response models, OllamaProvider, GeminiProvider, factory functions.

In [1]:
import sys
sys.path.insert(0, "../..")

## 1. Response Models

In [2]:
from src.services.llm.provider import UsageMetadata, LLMResponse, HealthStatus

usage = UsageMetadata(prompt_tokens=10, completion_tokens=20, total_tokens=30, latency_ms=150.5)
assert usage.total_tokens == 30

resp = LLMResponse(text="Hello", model="llama3.2", provider="ollama", usage=usage)
assert resp.provider == "ollama"
assert resp.usage.latency_ms == 150.5

hs = HealthStatus(healthy=True, provider="ollama", message="OK", version="0.5")
assert hs.healthy

print("\n\u2713 Response models validated: UsageMetadata, LLMResponse, HealthStatus")


✓ Response models validated: UsageMetadata, LLMResponse, HealthStatus


## 2. LLMProvider Protocol

In [3]:
from src.services.llm.provider import LLMProvider
from src.services.llm.ollama_provider import OllamaProvider
from src.services.llm.gemini_provider import GeminiProvider

# Both providers implement the protocol (runtime_checkable)
from src.config import OllamaSettings
ollama = OllamaProvider(OllamaSettings())
assert isinstance(ollama, LLMProvider), "OllamaProvider must implement LLMProvider"
print("OllamaProvider implements LLMProvider:", isinstance(ollama, LLMProvider))

print("\n\u2713 LLMProvider protocol is runtime_checkable and satisfied by providers")

OllamaProvider implements LLMProvider: True

✓ LLMProvider protocol is runtime_checkable and satisfied by providers


## 3. OllamaProvider (mocked)

In [4]:
from unittest.mock import AsyncMock, patch
import httpx

from src.services.llm.ollama_provider import OllamaProvider
from src.config import OllamaSettings

provider = OllamaProvider(OllamaSettings())

# Mock generate
mock_resp = httpx.Response(200, json={
    "response": "Test answer", "model": "llama3.2:1b",
    "prompt_eval_count": 5, "eval_count": 10, "total_duration": 200_000_000
})

with patch.object(provider._client, "post", new_callable=AsyncMock, return_value=mock_resp):
    result = await provider.generate("What is attention?")

assert result.text == "Test answer"
assert result.provider == "ollama"
assert result.usage.prompt_tokens == 5
print(f"  generate() -> text={result.text!r}, usage={result.usage}")

print("\n\u2713 OllamaProvider.generate() works correctly (mocked)")

  generate() -> text='Test answer', usage=prompt_tokens=5 completion_tokens=10 total_tokens=15 latency_ms=200.0

✓ OllamaProvider.generate() works correctly (mocked)


## 4. GeminiProvider (mocked)

In [5]:
from unittest.mock import MagicMock, patch
from src.services.llm.gemini_provider import GeminiProvider
from src.config import GeminiSettings

with patch("src.services.llm.gemini_provider.genai"):
    gp = GeminiProvider(GeminiSettings(api_key="test-key"))

mock_result = MagicMock()
mock_result.text = "Gemini response"
mock_result.usage_metadata = MagicMock(
    prompt_token_count=3, candidates_token_count=7, total_token_count=10
)
gp._client.models.generate_content = MagicMock(return_value=mock_result)

result = await gp.generate("What is BERT?")
assert result.text == "Gemini response"
assert result.provider == "gemini"
assert result.usage.total_tokens == 10
print(f"  generate() -> text={result.text!r}, usage={result.usage}")

print("\n\u2713 GeminiProvider.generate() works correctly (mocked)")

  generate() -> text='Gemini response', usage=prompt_tokens=3 completion_tokens=7 total_tokens=10 latency_ms=None

✓ GeminiProvider.generate() works correctly (mocked)


## 5. Factory Functions

In [6]:
from src.services.llm.factory import create_llm_provider, create_llm_providers
from src.config import Settings

# No Gemini key -> Ollama
settings = Settings()
settings.gemini = GeminiSettings(api_key="")
provider = create_llm_provider(settings)
assert isinstance(provider, OllamaProvider)
print(f"  No API key -> {type(provider).__name__}")

# With Gemini key -> Gemini
settings2 = Settings()
settings2.gemini = GeminiSettings(api_key="test-key")
with patch("src.services.llm.factory.genai"):
    provider2 = create_llm_provider(settings2)
assert isinstance(provider2, GeminiProvider)
print(f"  API key set -> {type(provider2).__name__}")

# Multi-provider
with patch("src.services.llm.factory.genai"):
    providers = create_llm_providers(settings2)
assert "ollama" in providers
assert "gemini" in providers
print(f"  Multi-provider keys: {list(providers.keys())}")

print("\n\u2713 Factory functions select correct provider based on config")

  No API key -> OllamaProvider
  API key set -> GeminiProvider
  Multi-provider keys: ['ollama', 'gemini']

✓ Factory functions select correct provider based on config


## 6. Error Handling

In [7]:
from src.exceptions import LLMConnectionError, LLMTimeoutError, ConfigurationError

# Missing API key raises ConfigurationError
try:
    GeminiProvider(GeminiSettings(api_key=""))
    assert False, "Should have raised"
except ConfigurationError as e:
    print(f"  Missing API key: {e.detail}")

# Connection error maps correctly
err_provider = OllamaProvider(OllamaSettings())

with patch.object(err_provider._client, "post", new_callable=AsyncMock, side_effect=httpx.ConnectError("refused")):
    try:
        await err_provider.generate("test")
        assert False
    except LLMConnectionError as e:
        print(f"  ConnectError -> LLMConnectionError: {e.detail}")

with patch.object(err_provider._client, "post", new_callable=AsyncMock, side_effect=httpx.TimeoutException("slow")):
    try:
        await err_provider.generate("test")
        assert False
    except LLMTimeoutError as e:
        print(f"  TimeoutException -> LLMTimeoutError: {e.detail}")

print("\n\u2713 Error mapping works correctly for all exception types")

  Missing API key: GEMINI__API_KEY is required for GeminiProvider
  ConnectError -> LLMConnectionError: Cannot connect to Ollama at http://localhost:11434: refused
  TimeoutException -> LLMTimeoutError: Ollama generation timed out: slow

✓ Error mapping works correctly for all exception types


## 7. DI Integration

In [8]:
from src.dependency import LLMProviderDep, get_llm_provider

assert callable(get_llm_provider)
print(f"  LLMProviderDep type alias: {LLMProviderDep}")
print(f"  get_llm_provider callable: {get_llm_provider}")

print("\n\u2713 LLM provider wired into FastAPI dependency injection")

  LLMProviderDep type alias: typing.Annotated[src.services.llm.provider.LLMProvider, Depends(dependency=<function get_llm_provider at 0x17ebd53a0>, use_cache=True, scope=None)]
  get_llm_provider callable: <function get_llm_provider at 0x17ebd53a0>

✓ LLM provider wired into FastAPI dependency injection


In [9]:
print("="*60)
print("S5.1 -- Unified LLM Client: ALL CHECKS PASSED")
print("="*60)

S5.1 -- Unified LLM Client: ALL CHECKS PASSED
